# CPT Notebook — Continued Pre-Training (Self-Contained)

**Semua logic sudah ada di dalam notebook ini. Tidak butuh file `.py` eksternal.**

### Fitur
- Real-time W&B monitoring
- Incremental data support — tambah CSV baru, rerun, model lanjut belajar
- Auto-resume dari checkpoint jika training terputus
- Pre-download model dengan resume support
- **TEST_MODE** — tes pipeline cepat dengan data terbatas

## Step 1: Configuration

In [ ]:
# ============================================
# CONFIGURATION — EDIT BAGIAN INI
# ============================================

# ── TEST MODE ──────────────────────────────
# Set True untuk tes pipeline cepat (~5 menit)
# Set False untuk full training
TEST_MODE = True
MAX_ROWS = 1000        # Jumlah dokumen maks saat TEST_MODE
TEST_EPOCHS = 1        # Jumlah epoch saat TEST_MODE
# ───────────────────────────────────────────

MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"

# W&B
WANDB_PROJECT = "tim1-dfk"
WANDB_RUN_NAME = "cpt-run-"
WANDB_ENTITY = None

# Training
NUM_EPOCHS = TEST_EPOCHS if TEST_MODE else 3
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 8  # effective batch = 2*8 = 16
LEARNING_RATE = 2e-4
BLOCK_SIZE = 2048
SAVE_STEPS = 100

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Paths (relative to Google Drive project folder)
DRIVE_PROJECT = "/content/drive/MyDrive/AITF/Tim1-DFK"
CPT_DATA_DIR = "Dataset/CPT"
CPT_OUTPUT_DIR = "outputs/cpt"

# Use existing LoRA for continual training?
USE_EXISTING_LORA = True

print("=" * 60)
if TEST_MODE:
    print(" TEST MODE AKTIF — pipeline test dengan data terbatas")
    print(f"  Max rows: {MAX_ROWS}")
else:
    print(" FULL TRAINING MODE")
print("=" * 60)
print(f"  Model: {MODEL_NAME}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Effective Batch Size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  LoRA r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  W&B Project: {WANDB_PROJECT}")
print(f"  Continue from existing LoRA: {USE_EXISTING_LORA}")
print("=" * 60)

## Step 2: Install Dependencies

In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q hf_transfer wandb rich tqdm pandas

## Step 3: Setup Environment

In [ ]:
import os, sys, re, json, hashlib, logging
from pathlib import Path
from datetime import datetime

# HF cache ke local disk (lebih cepat dari Google Drive)
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs('/content/hf_cache', exist_ok=True)
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path(DRIVE_PROJECT)
if not PROJECT_ROOT.exists():
    print(f'ERROR: {DRIVE_PROJECT} tidak ditemukan di Google Drive!')
    sys.exit(1)

os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')
print(f'Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## Step 4: Setup W&B

In [ ]:
import wandb

wandb.login()

api = wandb.Api()
viewer = api.viewer
username = viewer.entity if hasattr(viewer, 'entity') else str(viewer)
print(f'W&B connected as: {username}')

os.environ['WANDB_API_KEY'] = wandb.api.api_key
if WANDB_ENTITY:
    os.environ['WANDB_ENTITY'] = WANDB_ENTITY

timestamp = datetime.now().strftime("%m%d_%H%M%S")
WANDB_RUN_FULL = f"{WANDB_RUN_NAME}{timestamp}"
entity_display = WANDB_ENTITY or username
print(f'Run name: {WANDB_RUN_FULL}')
print(f'Monitor at: https://wandb.ai/{entity_display}/{WANDB_PROJECT}')

## Step 5: Pre-Download Model

Download model terlebih dahulu agar bisa di-resume jika terputus.
Jika download terhenti, **jalankan cell ini lagi**.

In [ ]:
from huggingface_hub import snapshot_download
import shutil

MODEL_LOCAL = "/content/hf_cache/" + MODEL_NAME.split("/")[-1]

existing = list(Path(MODEL_LOCAL).rglob("*.safetensors")) if Path(MODEL_LOCAL).exists() else []
if existing:
    total_gb = sum(f.stat().st_size for f in existing) / 1e9
    print(f"Model sudah ter-cache: {total_gb:.2f} GB ({len(existing)} files)")
else:
    disk = shutil.disk_usage("/content")
    print(f"Disk tersedia: {disk.free / 1e9:.1f} GB")
    print(f"Downloading {MODEL_NAME}...")
    snapshot_download(repo_id=MODEL_NAME, local_dir=MODEL_LOCAL, resume_download=True)
    files = list(Path(MODEL_LOCAL).rglob("*.safetensors"))
    total_gb = sum(f.stat().st_size for f in files) / 1e9
    print(f"Model downloaded: {total_gb:.2f} GB")

## Step 6: Preprocess CPT Data

Memuat semua CSV dari `Dataset/CPT/`, membersihkan teks, deduplikasi, dan menyimpan ke corpus.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

MIN_TEXT_LENGTH = 50

# ── Fungsi Preprocessing (inline) ──

def clean_text_minimal(text):
    """Minimal cleaning — hanya hapus HTML dan normalize whitespace."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_texts_from_csv(filepath):
    """Extract teks dari CSV, auto-detect kolom teks."""
    try:
        df = pd.read_csv(filepath, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(filepath, encoding="latin-1")

    print(f"  Loaded {filepath.name}: {len(df)} rows, columns: {list(df.columns)}")

    text_col = None
    for col in ["text", "content", "teks", "isi", "artikel", "body"]:
        if col in df.columns:
            text_col = col
            break

    if text_col is None:
        str_cols = df.select_dtypes(include=["object"]).columns
        if len(str_cols) > 0:
            avg_lens = {c: df[c].dropna().astype(str).str.len().mean() for c in str_cols}
            text_col = max(avg_lens, key=avg_lens.get)
            print(f"  Auto-detected text column: '{text_col}' (avg len: {avg_lens[text_col]:.0f})")
        else:
            print(f"  WARNING: No text column found in {filepath.name}, skipping.")
            return []

    texts = []
    for _, row in df.iterrows():
        text = str(row[text_col]) if pd.notna(row[text_col]) else ""
        text = clean_text_minimal(text)
        if len(text) >= MIN_TEXT_LENGTH:
            texts.append(text)
    return texts


def deduplicate_texts(texts):
    """Hapus duplikat menggunakan MD5 hash."""
    seen = set()
    unique = []
    for text in texts:
        normalized = re.sub(r"\s+", " ", text.lower().strip())
        h = hashlib.md5(normalized.encode("utf-8")).hexdigest()
        if h not in seen:
            seen.add(h)
            unique.append(text)
    return unique


# ── Pipeline ──

DATA_DIR = PROJECT_ROOT / CPT_DATA_DIR
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted([f for f in DATA_DIR.glob("*.csv") if f.parent == DATA_DIR])
print(f"CSV files ditemukan: {len(csv_files)}")

all_texts = []
for csv_file in csv_files:
    print(f"\nProcessing: {csv_file.name}")
    texts = extract_texts_from_csv(csv_file)
    print(f"  Extracted {len(texts)} valid texts")
    all_texts.extend(texts)

print(f"\nTotal sebelum dedup: {len(all_texts)}")
unique_texts = deduplicate_texts(all_texts)
removed = len(all_texts) - len(unique_texts)
print(f"Duplikat dihapus: {removed}")
print(f"Total setelah dedup: {len(unique_texts)}")

# ── Apply MAX_ROWS jika TEST_MODE ──
if TEST_MODE and len(unique_texts) > MAX_ROWS:
    print(f"\n TEST_MODE: Membatasi dari {len(unique_texts)} → {MAX_ROWS} dokumen")
    unique_texts = unique_texts[:MAX_ROWS]

# Simpan corpus
corpus_file = OUT_DIR / "cpt_corpus.txt"
with open(corpus_file, "w", encoding="utf-8") as f:
    for text in unique_texts:
        f.write(text.replace("\n", " ").replace("\r", " ") + "\n")

# Simpan stats
word_counts = [len(t.split()) for t in unique_texts]
stats = {
    "total_documents": len(unique_texts),
    "total_words": sum(word_counts),
    "avg_words_per_doc": round(sum(word_counts) / max(len(unique_texts), 1), 1),
    "min_words": min(word_counts) if word_counts else 0,
    "max_words": max(word_counts) if word_counts else 0,
    "duplicates_removed": removed,
    "source_files": [f.name for f in csv_files],
    "test_mode": TEST_MODE,
}
with open(OUT_DIR / "cpt_stats.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f"\nCorpus saved: {corpus_file} ({corpus_file.stat().st_size / 1024:.1f} KB)")
print(f"Documents: {stats['total_documents']:,}")
print(f"Total words: {stats['total_words']:,}")
print(f"Avg words/doc: {stats['avg_words_per_doc']}")

## Step 7: CPT Training

Training causal language modeling dengan Unsloth + LoRA.
Jika terputus, jalankan cell ini lagi — auto-resume dari checkpoint terakhir.

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# ── Load Model ──
print("Loading model...")

model_name = MODEL_NAME
local_path = "/content/hf_cache/" + MODEL_NAME.split("/")[-1]
if os.path.exists(local_path):
    model_name = local_path
    print(f"  Using cached model: {local_path}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=BLOCK_SIZE,
    load_in_4bit=True,
    dtype=None,
    device_map="auto",
)
print(f"Model loaded: {model_name}")

# ── Check Existing LoRA ──
lora_adapter_path = PROJECT_ROOT / CPT_OUTPUT_DIR / "lora_adapter"
has_existing_lora = USE_EXISTING_LORA and lora_adapter_path.exists()

if has_existing_lora:
    print(f"\nExisting CPT LoRA found: {lora_adapter_path}")
    print("Loading for continued training...")
    try:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(lora_adapter_path), is_trainable=True)
        print("Existing LoRA loaded!")
    except Exception as e:
        print(f"WARNING: Failed to load LoRA: {e}")
        print("Applying fresh LoRA instead...")
        has_existing_lora = False

if not has_existing_lora:
    print("\nApplying fresh LoRA adapter...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none",
        use_rslora=True,
        use_gradient_checkpointing="unsloth",
    )

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# ── Load Corpus ──
corpus_path = PROJECT_ROOT / CPT_DATA_DIR / "processed" / "cpt_corpus.txt"
with open(corpus_path, "r", encoding="utf-8") as f:
    texts = [line.strip() for line in f if line.strip()]
print(f"\nLoaded {len(texts):,} documents")

dataset = Dataset.from_dict({"text": texts})

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=BLOCK_SIZE, padding=False)

print("Tokenizing...")
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"], desc="Tokenizing")
print(f"Tokenized: {len(tokenized):,} examples")

# ── W&B Init ──
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_FULL,
    config={
        "model": MODEL_NAME, "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION,
        "lr": LEARNING_RATE, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "test_mode": TEST_MODE, "max_rows": MAX_ROWS if TEST_MODE else "all",
    },
    tags=["cpt", "qwen3", "dfk"] + (["test"] if TEST_MODE else []),
    reinit=True,
)

# ── Training ──
output_dir = PROJECT_ROOT / CPT_OUTPUT_DIR

training_args = TrainingArguments(
    output_dir=str(output_dir),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    seed=42,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    logging_steps=10,
    report_to="wandb",
    optim="adamw_8bit",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
)

# Auto-resume dari checkpoint (skip di TEST_MODE — selalu fresh)
last_ckpt = None
if not TEST_MODE and output_dir.exists():
    ckpts = sorted(output_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
    if ckpts:
        last_ckpt = str(ckpts[-1])
        print(f"Resuming from checkpoint: {last_ckpt}")

if TEST_MODE:
    print(f"\n⚡ TEST_MODE: {NUM_EPOCHS} epoch, {len(tokenized)} samples (fresh, no checkpoint resume)")
print("\nStarting CPT training...")
trainer.train(resume_from_checkpoint=last_ckpt)
print("\nTraining complete!")

# ── Save LoRA Adapter ──
lora_save = str(output_dir / "lora_adapter")
os.makedirs(lora_save, exist_ok=True)
model.save_pretrained(lora_save)
tokenizer.save_pretrained(lora_save)
print(f"LoRA adapter saved: {lora_save}")

wandb.finish()

## Step 8: Verify Output

In [ ]:
lora_path = PROJECT_ROOT / CPT_OUTPUT_DIR / "lora_adapter"

if lora_path.exists():
    print("CPT TRAINING COMPLETE!")
    print("=" * 60)
    print(f"LoRA adapter: {lora_path}")
    total_size = 0
    for f in sorted(lora_path.glob("*")):
        size_kb = f.stat().st_size / 1024
        total_size += size_kb
        print(f"  {f.name} ({size_kb:.1f} KB)")
    print(f"  Total: {total_size/1024:.1f} MB")
    print("")
    if TEST_MODE:
        print("⚡ Ini hasil TEST MODE — untuk full training, set TEST_MODE = False")
    print("\nNext: Buka 2_SFT_Colab.ipynb untuk SFT training.")
    print("SFT akan otomatis memuat CPT LoRA ini.")
    print("Model final (merged 16-bit) akan dibuat di akhir SFT notebook.")
else:
    print("ERROR: LoRA adapter tidak ditemukan!")
    print("Training mungkin gagal. Cek output di atas.")

---
## Summary

### Output
```
outputs/cpt/
└── lora_adapter/
    ├── adapter_config.json
    ├── adapter_model.safetensors
    └── tokenizer files
```

CPT menghasilkan LoRA adapter (intermediate). Model final 16-bit yang merged akan dibuat di **2_SFT_Colab.ipynb** dan disimpan di Google Drive.

### Test Mode vs Full Training
| Setting | Test Mode | Full Training |
|---------|-----------|---------------|
| `TEST_MODE` | `True` | `False` |
| `MAX_ROWS` | 1000 | semua data |
| `NUM_EPOCHS` | 1 | 3 |
| Waktu | ~5 menit | 1.5-3 jam |

### Incremental Updates
1. Tambah CSV baru ke `Dataset/CPT/`
2. Jalankan ulang Step 6 (preprocess)
3. Jalankan ulang Step 7 (training)
4. Model akan lanjut dari LoRA yang sudah ada

### Next Step
Lanjut ke **2_SFT_Colab.ipynb** untuk Supervised Fine-Tuning → merge model final.